# NEAR MAG over selected orbit windows

This notebook loads the full calibrated MAG time series once, then creates interactive 3D Plotly views for smaller windows representing different Eros orbital-radius regimes. Positions are geometric J2000 coordinates of NEAR relative to Eros from the archived SPICE kernels. Marker color shows the selected magnetic-field component.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
import spiceypy as spice

In [ ]:
data_directory = Path("/Users/danywaller/Projects/near/data")
color_component = "btotal"
maximum_points = 20000
gap_seconds = 1800.0
marker_size = 3.0

components = {"bx": 0, "by": 1, "bz": 2, "btotal": 3}
labels = {
    "bx": "Bx (nT)",
    "by": "By (nT)",
    "bz": "Bz (nT)",
    "btotal": "|B| (nT)",
}

In [ ]:
def parse_time(value):
    parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)
    return parsed.astimezone(timezone.utc)


def read_mag(data_directory):
    files = sorted((data_directory / "mag").rglob("*.tab"))
    if not files:
        raise FileNotFoundError(f"no calibrated MAG tables below {data_directory / 'mag'}")
    frames = {path.name[:3].lower() for path in files}
    if len(frames) != 1 or not frames <= {"nso", "ebf"}:
        raise ValueError("the MAG directory must contain one coordinate frame")
    time_chunks = []
    field_chunks = []
    for path in files:
        raw_times = np.loadtxt(path, delimiter=",", usecols=0, dtype="U32", ndmin=1)
        fields = np.loadtxt(path, delimiter=",", usecols=(6, 7, 8, 9), ndmin=2)
        times = np.array(
            [np.datetime64(value, "ms") for value in raw_times],
            dtype="datetime64[ms]",
        )
        time_chunks.append(times)
        field_chunks.append(fields)
    times = np.concatenate(time_chunks)
    fields = np.concatenate(field_chunks)
    order = np.argsort(times)
    return times[order], fields[order], frames.pop().upper()


def load_spice(spice_directory):
    kernels = sorted(spice_directory.rglob("*.tls")) + sorted(
        spice_directory.rglob("*.bsp")
    )
    if not kernels:
        raise FileNotFoundError(f"no SPICE kernels below {spice_directory}")
    spice.kclear()
    for kernel in kernels:
        spice.furnsh(str(kernel))


def select_window(start, stop):
    start = np.datetime64(parse_time(start).replace(tzinfo=None), "ms")
    stop = np.datetime64(parse_time(stop).replace(tzinfo=None), "ms")
    keep = (all_times >= start) & (all_times <= stop)
    if not np.any(keep):
        raise ValueError("no MAG samples fall inside this window")
    times = all_times[keep]
    fields = all_fields[keep]
    if times.size > maximum_points:
        indices = np.unique(
            np.linspace(0, times.size - 1, maximum_points, dtype=int)
        )
        times = times[indices]
        fields = fields[indices]
    return times, fields


def eros_positions(times):
    utc = [np.datetime_as_string(value, unit="ms") for value in times]
    positions = []
    for first in range(0, times.size, 10000):
        ephemeris_times = np.asarray(spice.str2et(utc[first : first + 10000]))
        chunk, _ = spice.spkpos(
            "-93", ephemeris_times, "J2000", "NONE", "2000433"
        )
        positions.append(chunk)
    return np.concatenate(positions)


def trajectory_with_gaps(times, positions):
    seconds = times.astype("datetime64[ms]").astype(np.int64) / 1000
    breaks = np.flatnonzero(np.diff(seconds) > gap_seconds) + 1
    coordinates = [positions[:, index].astype(object) for index in range(3)]
    for index in breaks[::-1]:
        coordinates = [np.insert(values, index, None) for values in coordinates]
    return coordinates


def plot_window(start, stop, title):
    if color_component not in components:
        raise ValueError(f"color_component must be one of {', '.join(components)}")
    times, fields = select_window(start, stop)
    positions = eros_positions(times)
    line_x, line_y, line_z = trajectory_with_gaps(times, positions)
    distance = np.linalg.norm(positions, axis=1)
    color = fields[:, components[color_component]]
    utc = [np.datetime_as_string(value, unit="ms") for value in times]
    hover = [
        "<br>".join(
            [
                f"UTC: {utc[index]}",
                f"Bx: {fields[index, 0]:.3f} nT",
                f"By: {fields[index, 1]:.3f} nT",
                f"Bz: {fields[index, 2]:.3f} nT",
                f"|B|: {fields[index, 3]:.3f} nT",
                f"distance: {distance[index]:.3f} km",
            ]
        )
        for index in range(times.size)
    ]
    figure = go.Figure()
    figure.add_trace(
        go.Scatter3d(
            x=line_x,
            y=line_y,
            z=line_z,
            mode="lines",
            line={"color": "rgba(70, 70, 70, 0.45)", "width": 1},
            hoverinfo="skip",
            name="NEAR trajectory",
        )
    )
    figure.add_trace(
        go.Scatter3d(
            x=positions[:, 0],
            y=positions[:, 1],
            z=positions[:, 2],
            mode="markers",
            marker={
                "size": marker_size,
                "color": color,
                "colorscale": "Turbo",
                "colorbar": {"title": labels[color_component]},
                "opacity": 0.85,
            },
            text=hover,
            hovertemplate="%{text}<extra></extra>",
            name="MAG samples",
        )
    )
    figure.add_trace(
        go.Scatter3d(
            x=[0],
            y=[0],
            z=[0],
            mode="markers",
            marker={"size": 7, "color": "#222222", "symbol": "diamond"},
            hovertemplate="Eros<extra></extra>",
            name="Eros",
        )
    )
    figure.update_layout(
        title=f"{title}<br><sup>{start} to {stop} UTC | {field_frame} field</sup>",
        template="plotly_white",
        scene={
            "xaxis_title": "J2000 X relative to Eros (km)",
            "yaxis_title": "J2000 Y relative to Eros (km)",
            "zaxis_title": "J2000 Z relative to Eros (km)",
            "aspectmode": "data",
        },
        legend={"orientation": "h", "y": 1.02, "x": 0},
        margin={"l": 0, "r": 0, "b": 0, "t": 90},
    )
    return figure

In [ ]:
all_times, all_fields, field_frame = read_mag(data_directory)
load_spice(data_directory / "spice")
print(f"loaded {all_times.size:,} {field_frame} MAG samples")
print(f"coverage: {all_times[0]} to {all_times[-1]}")

## 321 × 366 km insertion orbit

In [ ]:
plot_window(
    "2000-02-14T15:33:00",
    "2000-02-17T00:00:00",
    "Eros insertion orbit: 321 × 366 km radius",
)

## 203 × 206 km orbit

In [ ]:
plot_window(
    "2000-03-15T00:00:00",
    "2000-03-18T00:00:00",
    "Eros orbit: 203 × 206 km radius",
)

## 101 × 99 km orbit

In [ ]:
plot_window(
    "2000-04-15T00:00:00",
    "2000-04-17T00:00:00",
    "Eros orbit: 101 × 99 km radius",
)

## 51 × 49 km orbit

In [ ]:
plot_window(
    "2000-05-05T00:00:00",
    "2000-05-07T00:00:00",
    "Eros orbit: 51 × 49 km radius",
)

## 39 × 35 km orbit

In [ ]:
plot_window(
    "2000-07-16T00:00:00",
    "2000-07-17T00:00:00",
    "Eros orbit: 39 × 35 km radius",
)

## 51 × 19 km close orbit

In [ ]:
plot_window(
    "2000-10-26T00:00:00",
    "2000-10-26T17:30:00",
    "Eros close orbit: 51 × 19 km radius",
)